# GCM BA - 5. Dependence structure of F1

F1 is a quantile delta mapping applied **pixel by pixel and variable by variable**. Such a univariate correction can
change the dependence between days, between pixels and between variables (Vrac & Friederichs 2015, *J. Climate* 28,
218–237). The aim here is to measure it, on the EC-Earth3 grid over the LISFLOOD domain (European land), with:

1. **temporal**: lag-1 autocorrelation of pr and of the tasmax anomaly (monthly climatology removed), probabilities of
   a wet day after a wet / dry day (wet = pr ≥ 1 mm/day), mean wet and dry spells, CDD and CWD (mean annual maximum
   dry / wet spell);
2. **between variables**: Spearman correlation of daily pr and tasmax;
3. **spatial**: mean correlation of the daily series of every pair of pixels by distance (50 km classes), and the
   e-folding distance (where it falls below 1/e);
4. **areal extremes**, what a river basin integrates: Rx1day and Rx5day of the mean pr over blocks of 3 × 3 and
   6 × 6 cells (about 225 and 450 km), and the areal reduction factor ARF = areal Rx1day / mean Rx1day of its pixels.

DJF and JJA for the seasonal statistics. Three questions:
- **does F1 change the dependence of the model?** EC-Earth3 + F1 against raw EC-Earth3, same days (it should not much:
  QDM is monotonic in each pixel and month, so the order of the days of a pixel is kept);
- **how far is the model from ERA5?** raw and F1 against ERA5 1981–2014. EC-Earth3 is a free run: only statistics are
  compared (VALUE framework, Maraun et al. 2015, *Earth's Future* 3, 1–14). A dependence error of the model is not
  corrected by F1 and is not meant to be: imposing ERA5's would also replace the model's own change of it;
- **is the change of the dependence preserved?** climate change (original − historical) and AMOC collapse
  (hosing − original), raw against F1.

Statistics cached on EOS; figures and tables in `outputs/` next to this notebook.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
cfg = bu.load_config(CONFIG_PATH)
CAL_PERIOD = ("1981", "2014")      # ERA5 / EC-Earth3 historical, as F1
REGION = bu.gcm_ba_region(cfg, 1.0)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}" + ("_europe" if REGION else "")
OVERWRITE_CACHE = False
DOMAIN = vu.init(cfg, "domain", borders_res="01M")       # LISFLOOD domain without the land outside Europe
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")
CACHE = bu.gcm_ba_dir(cfg, "5_dependence", "cache" + ("_europe" if REGION else ""))
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
print(f"region {REGION} | F1 {F1_TAG} | cache {CACHE} | wet day >= {vu.WET_DEP} mm/day")

## 1. Statistics (computed once per dataset, cached)

In [ ]:
def series(name, adjusted):
    # pr and tasmax of a dataset on the domain pixels: ERA5 and historical over CAL_PERIOD, the runs whole
    period = CAL_PERIOD if name in ("era5", "gcm_historical") else None
    if adjusted:
        ds = xr.open_dataset(OUTF1 / f"{name}_{F1_TAG}.nc", chunks={"time": 3100}, **bu._TIME_KW)
        d = {v: ds[v] if period is None else bu.subset_time(ds[v], *period) for v in ("pr", "tasmax")}
    else:
        d = {v: bu.gcm_ba_input(cfg, name, v, period, REGION) for v in ("pr", "tasmax")}
    return {v: vu.domain_pixels(a) for v, a in d.items()}


DEP = {}
for name, adjusted, key in [("era5", False, "ERA5")] + [(n, a, f"{l} {'F1' if a else 'raw'}") for n, l in LABEL.items() for a in (False, True)]:
    if adjusted and not (OUTF1 / f"{name}_{F1_TAG}.nc").exists():
        print(f"[missing] {key}: run notebook 3"); continue
    path = CACHE / f"{name}_{F1_TAG if adjusted else 'raw'}_dependence.pkl"
    if path.exists() and not OVERWRITE_CACHE:
        DEP[key] = pd.read_pickle(path)
        continue
    s = series(name, adjusted)
    DEP[key] = vu.dependence_stats(s["pr"], s["tasmax"], path, OVERWRITE_CACHE, label=key)
    del s
W = np.cos(np.deg2rad(DEP["ERA5"]["lat"]))       # area weights of the domain pixels (same pixels in every dataset)
wmean = lambda a: float(np.nansum(W * a) / np.sum(W * np.isfinite(a)))
print("datasets:", list(DEP))

## 2. Historical climate (1981–2014): ERA5, raw EC-Earth3, EC-Earth3 + F1

For each statistic: the domain mean (cos-latitude weights) of ERA5, raw and F1; **F1 vs raw**, the mean absolute
difference and the pattern correlation of the two maps (what F1 changes in the model); **bias** of raw and F1
against ERA5 (what the model gets wrong, before and after F1).

In [ ]:
E, R, A = (DEP[k]["temporal"] for k in ("ERA5", "historical raw", "historical F1"))
rows = []
for c in E.columns:
    e, r, a = E[c].values, R[c].values, A[c].values
    ok = np.isfinite(r) & np.isfinite(a)
    rows.append({"statistic": c, "ERA5": wmean(e), "raw": wmean(r), "F1": wmean(a),
                 "F1 vs raw: mean |diff|": wmean(np.abs(a - r)), "F1 vs raw: pattern corr": np.corrcoef(r[ok], a[ok])[0, 1],
                 "bias raw": wmean(r - e), "bias F1": wmean(a - e)})
temporal = pd.DataFrame(rows).set_index("statistic")
temporal.to_csv(OUT / f"dependence_temporal_{F1_TAG}.csv")

sp = {k: DEP[k]["spatial"] for k in ("ERA5", "historical raw", "historical F1")}
spatial = pd.DataFrame({k: {f"{v}, {s}": vu.efold_km(d.index.values, d[(v, s)].values) for (v, s) in d.columns}
                        for k, d in sp.items()})
spatial.index.name = "e-folding distance (km)"
spatial.to_csv(OUT / f"dependence_spatial_{F1_TAG}.csv")

rows = []
for k in vu.DEP_BLOCKS:
    ar = {n: DEP[n]["areal"][k] for n in ("ERA5", "historical raw", "historical F1")}
    for stat in ("Rx1day", "Rx5day", "ARF"):
        if k == 1 and stat == "ARF":
            continue
        rows.append({"block": f"{k} x {k} cells", "statistic": stat, **{n: float(np.median(d[stat])) for n, d in ar.items()}})
areal = pd.DataFrame(rows).set_index(["block", "statistic"])
areal.to_csv(OUT / f"dependence_areal_{F1_TAG}.csv")
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print("(1) temporal and between variables (domain means)"); display(temporal.round(3))
    print("(2) spatial: e-folding distance of the mean correlation (km)"); display(spatial.round(0))
    print("(3) areal extremes: median over the blocks (mm/day, mm / 5 days; ARF)"); display(areal.round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
cols = list(sp["ERA5"].columns)
for ax, col in zip(axes, cols):
    for k, st in (("ERA5", "k-"), ("historical raw", "C0--"), ("historical F1", "C3-")):
        ax.plot(sp[k].index, sp[k][col], st, label=k)
    ax.axhline(np.exp(-1), color="0.6", lw=0.8)
    ax.set_title(f"{col[0]}, {col[1]}", fontsize=9); ax.set_xlabel("distance (km)"); ax.set_ylim(-0.1, 1)
axes[0].set_ylabel("mean correlation"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "dependence_spatial_correlation.png", dpi=90); plt.show()

grid = vu.mask_for(bu.gcm_ba_input(cfg, "gcm_historical", "tasmax", ("1981", "1981"), REGION).isel(time=0))
def to_map(v):
    m = xr.full_like(grid, np.nan, dtype=float)
    m.values[DEP["ERA5"]["ii"], DEP["ERA5"]["jj"]] = v
    return m
show = ["lag-1 autocorrelation pr, JJA", "P(wet | wet), JJA", "CDD (days)", "Spearman pr - tasmax, JJA"]
fig, axes = vu.map_axes(len(show), 4)
for r_, c in enumerate(show):
    e, r, a = E[c].values, R[c].values, A[c].values
    lim = (float(np.nanpercentile(e, 2)), float(np.nanpercentile(e, 98)))
    for k, (lab, v) in enumerate((("ERA5", e), ("raw", r), ("F1", a))):
        vu.draw(axes[r_, k], to_map(v), f"{c}: {lab}", cmap="viridis", lim=lim)
    vu.draw(axes[r_, 3], to_map(a - r), f"{c}: F1 - raw", cmap="RdBu_r", sym=True)
plt.tight_layout(); plt.savefig(OUT / "dependence_maps_historical.png", dpi=80); plt.show()

## 3. Is the change of the dependence preserved?

For each pair (climate change: original − historical; AMOC collapse: hosing − original): the domain-mean change of
each statistic, raw and F1, and the pattern regression of the F1 change on the raw change across pixels (slope 1 and
correlation 1 = preserved). Spatial: change of the e-folding distance. Areal: change of the median block Rx1day and
Rx5day (%).

In [ ]:
def slope_corr(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    w = W[ok]; x, y = x[ok], y[ok]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    vx, vy = np.average((x - mx) ** 2, weights=w), np.average((y - my) ** 2, weights=w)
    cov = np.average((x - mx) * (y - my), weights=w)
    return (cov / vx if vx > 0 else np.nan), (cov / np.sqrt(vx * vy) if vx > 0 and vy > 0 else np.nan)

rows = []
for a_, b_, lab in PAIRS:
    ka, kb = LABEL[a_], LABEL[b_]
    if any(f"{k} {t}" not in DEP for k in (ka, kb) for t in ("raw", "F1")):
        continue
    for c in E.columns:
        dr = DEP[f"{ka} raw"]["temporal"][c].values - DEP[f"{kb} raw"]["temporal"][c].values
        da = DEP[f"{ka} F1"]["temporal"][c].values - DEP[f"{kb} F1"]["temporal"][c].values
        s, r = slope_corr(dr, da)
        rows.append({"pair": lab, "statistic": c, "change raw": wmean(dr), "change F1": wmean(da), "pattern slope": s, "pattern corr": r})
    for (v, s_) in DEP["ERA5"]["spatial"].columns:
        ef = {t: vu.efold_km(DEP[f"{ka} {t}"]["spatial"].index.values, DEP[f"{ka} {t}"]["spatial"][(v, s_)].values)
              - vu.efold_km(DEP[f"{kb} {t}"]["spatial"].index.values, DEP[f"{kb} {t}"]["spatial"][(v, s_)].values) for t in ("raw", "F1")}
        rows.append({"pair": lab, "statistic": f"e-folding distance {v}, {s_} (km)", "change raw": ef["raw"], "change F1": ef["F1"]})
    for k in vu.DEP_BLOCKS[1:]:
        for stat in ("Rx1day", "Rx5day"):
            ch = {t: 100 * (np.median(DEP[f"{ka} {t}"]["areal"][k][stat]) / np.median(DEP[f"{kb} {t}"]["areal"][k][stat]) - 1) for t in ("raw", "F1")}
            rows.append({"pair": lab, "statistic": f"areal {stat}, {k} x {k} cells (%)", "change raw": ch["raw"], "change F1": ch["F1"]})
signals = pd.DataFrame(rows).set_index(["pair", "statistic"])
signals.to_csv(OUT / f"dependence_signals_{F1_TAG}.csv")
with pd.option_context("display.width", 250, "display.max_rows", 200):
    display(signals.round(3))

## 4. Reading the results

- **F1 vs raw** close (small mean |diff|, pattern correlation near 1) and **signal slopes near 1**: F1 keeps the
  dependence structure of EC-Earth3 and its change; a multivariate correction is not needed for that purpose.
- **Bias against ERA5** of the same size for raw and F1: it is the model's own error (resolution, convection), which
  a univariate correction neither creates nor removes. Imposing the dependence of ERA5 (MBCn, Cannon 2018; review in
  François et al. 2020, *Earth Syst. Dynam.* 11, 537–562) would correct it but also overwrite the change of the
  dependence simulated under the AMOC collapse.
- **Areal extremes**: what the basins of LISFLOOD integrate; a pixel-wise correction raises the pixel extremes, the
  ARF tells whether the areal extremes follow.